# Lezione 11 — Pandas: Series e DataFrame

## Obiettivi della lezione

- Capire cos'è una **Series** e cos'è un **DataFrame**, e in cosa si differenziano.
- Creare un DataFrame a partire da un dizionario, e salvarlo/caricarlo con `to_csv()` / `read_csv()`.
- Selezionare colonne con parentesi quadra singola (dà una **Series**) vs parentesi quadra doppia (dà un **DataFrame**).
- Padroneggiare la differenza tra `.loc` (per etichetta) e `.iloc` (per posizione), soprattutto nello slicing.
- Fare il "primo contatto" con un DataFrame nuovo: `head()`, `info()`, `describe()`, `shape`.

Lavoreremo con una versione del dataset **TopSellingAlbums** (i dischi più venduti della storia: Michael Jackson, AC/DC, Pink Floyd...). Invece di scaricarlo da internet, lo **creeremo direttamente nel notebook** con un dizionario, per non dipendere da nessuna connessione.


## 1. Cos'è una Series?

Una **Series** è un array a una sola dimensione, con una differenza chiave rispetto a una lista di Python: ogni valore ha un'etichetta associata, chiamata **indice**. Pensala come una colonna di un foglio di calcolo con la propria numerazione di righe attaccata di fianco.

Se non le assegni indici propri, pandas assegna automaticamente `0, 1, 2, 3, 4...`.


In [ ]:
import pandas as pd

# Creare una Series a partire da una lista
ventas = [46, 26.1, 24.2, 26.1, 20.6]
s = pd.Series(ventas)
print(s)


Con un indice di default (0,1,2,3...), accedere per **etichetta** e per **posizione** dà lo stesso risultato — questa coincidenza è proprio la trappola da imparare a evitare (più avanti, con `.loc`/`.iloc` su un DataFrame, vedrai perché la differenza conta).


In [ ]:
print(s[2])         # Accede per etichetta (indice 2) -> 24.2
print(s.iloc[3])    # Accede per posizione (posizione 3) -> 26.1
print(s[1:4])        # Intervallo di elementi per etichetta


Una Series porta con sé attributi e metodi incorporati per esplorare e riassumere i dati senza scrivere un ciclo. Ricorda il nome `apply()`: è il tuo strumento per applicare qualsiasi trasformazione personalizzata a ogni valore.


In [ ]:
print(s.values)        # il contenuto come array di NumPy
print(s.index)         # le etichette dell'indice
print(s.shape)         # dimensioni, per esempio (5,)
print(s.mean())        # media
print(s.sum())         # somma totale
print(s.unique())      # valori distinti, senza ripetizioni
print(s.sort_values()) # ordina la Series in base ai suoi valori
print(s.isnull())      # True/False a seconda che ogni valore sia nullo (NaN)
print(s.apply(lambda x: x * 1_000_000))  # applica una funzione a ogni elemento


## 2. Cos'è un DataFrame?

Se una Series è una colonna, un **DataFrame** è la tabella completa: una struttura bidimensionale, con righe e colonne etichettate, dove ogni colonna può avere il proprio tipo di dato. È l'oggetto che manipolerai di più in qualsiasi progetto di dati con Python.

Il modo più diretto per crearne uno a mano è a partire da un **dizionario**: ogni chiave diventa il nome di una colonna, e ogni valore (una lista) diventa i dati di quella colonna, riga per riga.


In [ ]:
datos = {
    'Artist': ['Michael Jackson', 'AC/DC', 'Pink Floyd'],
    'Album': ['Thriller', 'Back in Black', 'The Dark Side of the Moon'],
    'Released': [1982, 1980, 1973],
}
df_mini = pd.DataFrame(datos)
df_mini


Pandas allinea automaticamente ogni lista come una colonna e numera le righe partendo da 0. Quel numero è l'indice del DataFrame, e per default è puramente posizionale — più sotto vedrai come cambiarlo con `set_index()`.


## 3. Caricare dati reali: `read_csv()` e `read_excel()`

Nella pratica, quasi mai scriverai un DataFrame a mano: lo caricherai da un file, con una funzione per formato:

```python
df = pd.read_csv('TopSellingAlbums.csv')
df = pd.read_excel('TopSellingAlbums.xlsx')
```

In entrambi i casi il risultato è lo stesso tipo di oggetto: un DataFrame.

**Nota:** per non dipendere dallo scaricare un file da internet (potrebbe fallire senza connessione), **creeremo** il DataFrame completo con un dizionario — riproducendo i dati reali di `TopSellingAlbums` — e poi lo salveremo e lo rileggeremo con `to_csv()` / `read_csv()`, per praticare il flusso reale di lettura di un CSV senza dipendere dalla rete.


In [ ]:
datos_albumes = {
    'Artist': ['Michael Jackson', 'AC/DC', 'Pink Floyd', 'Whitney Houston',
               'Meat Loaf', 'Eagles', 'Bee Gees', 'Fleetwood Mac'],
    'Album': ['Thriller', 'Back in Black', 'The Dark Side of the Moon', 'The Bodyguard',
              'Bat Out of Hell', 'Their Greatest Hits (1971-1975)', 'Saturday Night Fever', 'Rumours'],
    'Released': [1982, 1980, 1973, 1992, 1977, 1976, 1977, 1977],
    'Length': ['00:42:19', '00:42:11', '00:42:49', '00:57:44',
               '00:46:33', '00:43:08', '01:15:54', '00:40:01'],
    'Genre': ['Pop, rock, R&B', 'Hard rock', 'Progressive rock', 'Soundtrack/R&B, soul, pop',
              'Hard rock, progressive rock', 'Rock, soft rock, folk rock', 'Disco', 'Soft rock'],
    'Sales': [46, 26.1, 24.2, 26.1, 20.6, 32.2, 20.6, 27.9],
    'Rating': [10.0, 8.5, 9.5, 7.0, 7.0, 9.5, 9.0, 9.5],
}

df = pd.DataFrame(datos_albumes)
df


Ora simuliamo il flusso reale: salviamo il DataFrame come file `.csv` su disco, e lo ricarichiamo con `read_csv()`, esattamente come faresti con un file che ti fosse stato consegnato.


In [ ]:
# Salviamo il DataFrame come se fosse il file che "riceviamo" da un'altra fonte
df.to_csv('TopSellingAlbums.csv', index=False)

# ...e lo ricarichiamo con read_csv(), come faresti con qualsiasi file reale
df = pd.read_csv('TopSellingAlbums.csv')
df

# Se il file fosse un Excel, lo schema è identico (richiede il pacchetto openpyxl):
# df = pd.read_excel('TopSellingAlbums.xlsx')


Il primo comando che dovresti scrivere dopo qualsiasi `read_csv()` è `head()`: ti conferma che il caricamento ha funzionato e ti dà un'idea immediata di come sono fatti i dati.


In [ ]:
df.head()      # prime 5 righe


In [ ]:
df.head(3)     # prime 3 righe, se ne vuoi meno


In [ ]:
df.tail(2)     # le ultime 2 righe


## 4. Selezionare colonne: Series o DataFrame?

Qui compare la prima fonte di confusione, e vale la pena interiorizzarla subito: **il numero di parentesi quadre cambia il tipo di risultato.**

- `df['col']` (una parentesi quadra) → una **Series**.
- `df[['col']]` (parentesi quadre doppie, con una lista dentro) → un **DataFrame** di una sola colonna.
- `df[['col1', 'col2']]` (lista con più nomi) → un **DataFrame** con quelle colonne, nell'ordine richiesto.


In [ ]:
x = df['Length']
print(type(x))   # <class 'pandas.core.series.Series'>
x


In [ ]:
x = df[['Length']]
print(type(x))   # <class 'pandas.core.frame.DataFrame'>
x


In [ ]:
y = df[['Artist', 'Length', 'Genre']]
y


Perché conta la differenza? Perché alcuni metodi si aspettano una Series e altri un DataFrame, e un errore di tipo qui di solito nasce dall'aver usato una parentesi quadra di troppo o di meno senza accorgertene. Quando hai dubbi, usa `type()`.


## 5. `.loc` vs `.iloc`: la differenza che confonde ogni principiante

Pandas ti dà due modi per accedere a righe (e celle specifiche) di un DataFrame:

- **`.iloc`** seleziona per **posizione intera**: "dammi la riga numero 2", indipendentemente da come si chiami. Funziona come l'indice di una lista di Python.
- **`.loc`** seleziona per **etichetta**: "dammi la riga chiamata 2" o "la riga chiamata 'Jane'", indipendentemente dalla posizione fisica in cui si trova.

| | `.iloc` (per posizione) | `.loc` (per etichetta) |
|---|---|---|
| Cosa si aspetta | Numeri interi di posizione | Nomi di riga/colonna (o numeri, se sono le etichette dell'indice) |
| Limite superiore dell'intervallo | **Esclusivo** — non include l'ultimo | **Inclusivo** — include anche l'ultimo |
| Esempio sulle righe | `df.iloc[0:2]` → righe 0 e 1 | `df.loc[0:2]` → righe 0, 1 **e** 2 |

Con un indice appena caricato (0,1,2,3... in ordine), entrambi danno lo stesso risultato per **una singola cella** — è lì che nasce la confusione, perché sembrano intercambiabili:


In [ ]:
print(df.iloc[0, 0])          # Valore nella riga 0, colonna 0 (per posizione)
print(df.loc[0, 'Artist'])    # Valore nella riga con etichetta 0, colonna 'Artist' (per nome)
# Con l'indice di default, entrambi restituiscono "Michael Jackson"

print(df.iloc[1, 2])          # Riga 1, colonna 2 (per posizione) -> 1980
print(df.loc[1, 'Released'])  # Riga con etichetta 1, colonna 'Released' -> 1980


Dove si separano davvero è nello **slicing** (gli intervalli con `:`). Confrontalo sullo stesso DataFrame, selezionando righe e colonne insieme:


In [ ]:
# Con iloc: righe 0 e 1 (la 2 resta fuori), colonne 0, 1 e 2 (la 3 resta fuori)
df.iloc[0:2, 0:3]


In [ ]:
# Con loc: righe con etichetta 0, 1 E 2 (il limite superiore VIENE incluso)
# colonne da 'Artist' fino a 'Released' inclusa
df.loc[0:2, 'Artist':'Released']


In [ ]:
print("Righe con df.iloc[0:2]:", len(df.iloc[0:2]))
print("Righe con df.loc[0:2]: ", len(df.loc[0:2]))
# Stesso intervallo scritto "0 a 2", numero di righe diverso


La differenza diventa ancora più visibile non appena l'indice smette di essere una semplice sequenza di numeri. Puoi sostituire l'indice numerico con etichette tue con `set_index()`:


In [ ]:
df2 = df.set_index('Artist')   # ora l'indice sono i nomi degli artisti, non 0,1,2...
df2.head()


In [ ]:
print(df2.loc['Michael Jackson', 'Released'])   # per etichetta: funziona, cerca il nome
print(df2.iloc[0, 1])                            # per posizione: continua a funzionare come prima


Una volta che l'indice è fatto di nomi, `.iloc` non se ne accorge nemmeno — continua a contare posizioni da 0 — mentre `.loc` ora richiede le nuove etichette, non i vecchi numeri. Se provi `df2.loc[0, 'Released']`, otterrai un errore, perché `0` non è più un'etichetta valida dell'indice:


In [ ]:
try:
    df2.loc[0, 'Released']
except KeyError as e:
    print("Errore atteso, perché 0 non è più un'etichetta dell'indice di df2:")
    print(e)


**Regola pratica per non confonderti mai più:** `.iloc` conta come una lista di Python (il limite superiore resta fuori). `.loc` cerca come in un dizionario o in un indice di libro (il limite superiore viene incluso, perché è un nome, non un conteggio).


## 6. Esplorazione rapida: `head()`, `info()`, `describe()`, `shape`

Prima di filtrare, raggruppare o graficare qualsiasi cosa, il tuo primo minuto con qualsiasi DataFrame nuovo dovrebbe andare in questi comandi:

- `head()` / `tail()`: prime/ultime righe — come sono fatti i dati?
- `info()`: quante righe e colonne ci sono, il tipo di dato di ogni colonna e quanti valori non nulli ha ciascuna.
- `describe()`: conteggio, media, deviazione standard, minimo, massimo e quartili di ogni colonna **numerica** (nota che `Length`, che è testo, non compare qui).
- `shape`: una tupla `(righe, colonne)` con la dimensione totale.


In [ ]:
df.info()


In [ ]:
df.describe()


In [ ]:
df.shape


## Riepilogo: cosa porti a casa da questa lezione

- Una **Series** è un array a una dimensione con un indice etichettato; una **colonna** di una tabella, con memoria delle proprie etichette.
- Un **DataFrame** è una tabella bidimensionale, con righe e colonne etichettate — l'oggetto centrale di pandas. Si crea da un dizionario (`pd.DataFrame()`) o, molto più spesso, da un file (`pd.read_csv()`, `pd.read_excel()`).
- `df['col']` (una parentesi quadra) restituisce una **Series**; `df[['col']]` o `df[['col1','col2']]` (parentesi quadre doppie) restituisce un **DataFrame**. Usa `type()` quando hai dubbi.
- **`.iloc[riga, colonna]`** seleziona per **posizione intera**, come una lista: il limite superiore di un intervallo resta **escluso**.
- **`.loc[riga, colonna]`** seleziona per **etichetta** (nome): il limite superiore di un intervallo viene **incluso**. Se cambi l'indice con `set_index()`, `.loc` deve usare le nuove etichette.
- `df.iloc[0:2, 0:3]` e `df.loc[0:2, 'A':'C']` non sono intercambiabili anche se sembrano simili: controlla sempre se il limite superiore entra o no nel risultato.
- `head()`, `tail()`, `info()`, `describe()` e `shape` sono il tuo rituale dei primi 60 secondi con qualsiasi DataFrame nuovo.
